# Window Functions

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

Two claims under test, each checked directly against real evidence rather than assumed:
1. `row_number()` over `partitionBy("user_id").orderBy("ts")`, plus a running total via `.rowsBetween(Window.unboundedPreceding, 0)`, produce a `Window` plan node preceded by the `Sort`/`Exchange` a windowed aggregation needs -- and the results are actually correct (verified against a hand-computed baseline, not just eyeballed).
2. The same query with `partitionBy` mistakenly dropped funnels the *entire* dataset onto a single task -- the plan node *shapes* look the same (`Window`/`Sort`/`Exchange` either way), so this notebook checks the Stages-tab task count directly, not the plan text.

In [ ]:
import json
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("window-functions").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def stages_snapshot():
    """Current `/api/v1/applications/<id>/stages` list -- the ground truth
    for 'how many tasks did this stage actually run', the same REST surface
    the Spark UI's Stages tab reads. Used below for the missing-partitionBy
    failure mode: no plan-node shape distinguishes 'correct' from
    'accidentally global' (see manifest.yaml's header comment), so this is
    checked directly against stage/task data instead -- same disposition as
    dag-lazy-evaluation's jobs_snapshot() and caching-persistence's
    storage_snapshot() helpers."""
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))

## 1. Build a keyed, timestamped dataset

Each user's rows are generated in ascending `ts` order, but spread across `NUM_INPUT_PARTITIONS` input partitions -- so `partitionBy("user_id").orderBy("ts")` genuinely has to shuffle rows for the same user back together and sort them, not just read them back in the order they were written.

In [ ]:
NUM_USERS = 2_000
EVENTS_PER_USER = 100
TOTAL_ROWS = NUM_USERS * EVENTS_PER_USER  # 200,000
NUM_INPUT_PARTITIONS = 24

rdd = spark.sparkContext.parallelize(range(TOTAL_ROWS), NUM_INPUT_PARTITIONS).map(
    lambda i: Row(
        user_id=f"user-{i % NUM_USERS}",
        ts=i // NUM_USERS,
        amount=float(i % 997),
    )
)
events_df = spark.createDataFrame(rdd)
print("Input partitions:", events_df.rdd.getNumPartitions())
print(f"Rows: {TOTAL_ROWS}  distinct users: {NUM_USERS}  events/user: {EVENTS_PER_USER}")
events_df.orderBy("user_id", "ts").show(5)

## 2. `row_number()` over `partitionBy("user_id").orderBy("ts")`

**Hypothesis:** for each user, what should the row numbered `1` be? How many rows total should have `rn == 1`? Write your answer before running the cell below.

In [ ]:
user_window = Window.partitionBy("user_id").orderBy("ts")
ranked_df = events_df.withColumn("rn", F.row_number().over(user_window))

row_count = ranked_df.count()
print(f"ranked_df.count() = {row_count} rows")

first_events = ranked_df.filter(F.col("rn") == 1).count()
print(f"Rows with rn == 1 (one per user, the first event in ts order): {first_events}")
assert first_events == NUM_USERS, (
    f"expected exactly {NUM_USERS} first-events (one per partition), got {first_events}"
)

## 3. Running total via `rowsBetween(Window.unboundedPreceding, 0)`

**Hypothesis:** at each user's *last* event (`rn == EVENTS_PER_USER`), what should `running_total` equal? Write your answer, then verify it below against an independently computed `groupBy("user_id").agg(F.sum("amount"))` baseline -- not just by eyeballing a few rows.

In [ ]:
running_window = user_window.rowsBetween(Window.unboundedPreceding, 0)
totals_df = ranked_df.withColumn("running_total", F.sum("amount").over(running_window))

last_events = totals_df.filter(F.col("rn") == EVENTS_PER_USER).select("user_id", "running_total")
full_sums = events_df.groupBy("user_id").agg(F.sum("amount").alias("total_amount"))

mismatches = (
    last_events.join(full_sums, "user_id")
    .filter(F.abs(F.col("running_total") - F.col("total_amount")) > 1e-6)
    .count()
)
print(f"Mismatches between running_total at the last event and the true per-user total: {mismatches}")
assert mismatches == 0, (
    "expected the running total at each partition's last row to equal the full per-user sum"
)

## 4. Read the physical plan -- `Window` preceded by `Sort`/`Exchange`

Look for the `Window` node, fed by a `Sort` (per-partition, per-`ts` ordering), fed by an `Exchange` (hash-partitioned shuffle on `user_id`) -- the same shape `concept.md`'s "What to look for" describes. Then confirm the stage actually doing that work spreads across many tasks, not just one.

In [ ]:
import contextlib
import io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    totals_df.explain(mode="formatted")
plan_text = buf.getvalue()
print(plan_text)

assert "Window" in plan_text, "expected a Window node in the physical plan"
assert "Sort" in plan_text, "expected a Sort node (per-partition ordering) preceding Window"
assert "Exchange" in plan_text, "expected an Exchange node (hash-partitioned shuffle on user_id) preceding Sort"

before = {s["stageId"] for s in stages_snapshot()}
# agg(sum("running_total")), not count() -- found by running this notebook for
# real: .count() lets Catalyst's column-pruning optimizer notice nothing
# downstream references "rn"/"running_total" and skip the Window/Sort/Exchange
# chain entirely (same class of pruning caching-persistence's notebook already
# flags for a plain UDF column). Aggregating "running_total" forces every row's
# window value to actually be computed.
totals_df.agg(F.sum("running_total")).collect()
new_stages = [s for s in stages_snapshot() if s["stageId"] not in before]
# Not max(stageId): this query also has a trailing single-task stage
# (collapsing the final sum's partial results into one scalar), and the
# map-side (pre-shuffle) input-read stage can itself have more tasks
# (NUM_INPUT_PARTITIONS) than a badly-partitioned window's reduce side --
# neither extreme is "the stage that ran the window". Only a stage fed by a
# shuffle (shuffleReadBytes > 0) is downstream of the Exchange/Sort/Window
# chain at all; among those, the one with the most tasks is the window's own
# reduce-side stage (the trailing single-task sum-collapse stage also has
# shuffleReadBytes > 0, but never more tasks than that).
shuffle_read_stages = [s for s in new_stages if s.get("shuffleReadBytes", 0) > 0]
window_stage = max(shuffle_read_stages, key=lambda s: s["numTasks"])
correct_num_tasks = window_stage["numTasks"]
print(f"\nStage {window_stage['stageId']}: numTasks={correct_num_tasks}")
assert correct_num_tasks > 1, "expected the correctly-partitioned window to spread across many tasks"

## 5. Checkpoint for the self-check reveal

Click **Reveal self-check** on the topic page after running this: the plan panel labels the `Window`/`Sort`/`Exchange` chain above.

In [ ]:
checkpoint(totals_df, topic="window-functions")
print("Checkpoint written -- click 'Reveal self-check' on the topic page.")

## 6. Missing `partitionBy` -- the accidental single-partition failure mode

**Hypothesis:** if `partitionBy("user_id")` is dropped and only `.orderBy("ts")` remains, does the plan look different? Does the task count? Write your answer, then check the driver container's log (`docker logs spark-driver`) for Spark's own `WARN` about this -- it logs "No Partition Defined for Window operation! Moving all data to a single partition" for exactly this case.

In [ ]:
bad_window = Window.orderBy("ts")  # no partitionBy -- falls back to one global order
bad_totals_df = events_df.withColumn(
    "running_total_bad", F.sum("amount").over(bad_window.rowsBetween(Window.unboundedPreceding, 0))
)

before = {s["stageId"] for s in stages_snapshot()}
bad_totals_df.agg(F.sum("running_total_bad")).collect()  # forces real computation -- see step 4's note
new_stages = [s for s in stages_snapshot() if s["stageId"] not in before]
shuffle_read_stages = [s for s in new_stages if s.get("shuffleReadBytes", 0) > 0]
bad_window_stage = max(shuffle_read_stages, key=lambda s: s["numTasks"])
bad_num_tasks = bad_window_stage["numTasks"]
print(f"Stage {bad_window_stage['stageId']}: numTasks={bad_num_tasks}")
assert bad_num_tasks == 1, (
    "expected the entire dataset funneled onto a single task -- check `docker logs spark-driver` for "
    "Spark's own 'Moving all data to a single partition' WARN for this same query"
)

## 7. Compare task counts side by side

The plan node *shapes* from step 4 and step 6 look the same (`Window`/`Sort`/`Exchange` either way) -- the task-count gap below is the real, measured difference a missing `partitionBy` causes.

In [ ]:
print(f"Correctly partitioned window: {correct_num_tasks} tasks")
print(f"Missing partitionBy window:   {bad_num_tasks} task(s)")
assert correct_num_tasks > bad_num_tasks, (
    "expected the correctly-partitioned window to use strictly more tasks than the missing-partitionBy one"
)